# Task 15: Distributed Data Parallel (DDP) Multi-GPU Node Synchronization

## Objective

Implement a PyTorch DistributedDataParallel training workflow for synchronized distributed training.

The implementation demonstrates:

- Process-group initialization
- DistributedDataParallel
- DistributedSampler
- Per-process GPU assignment
- Gradient synchronization
- Parameter synchronization
- Multi-process training
- Synchronization barriers
- Benchmarking distributed execution

DDP uses collective communication to synchronize gradients across workers.

For a parameter gradient \(g_i\):

\[
g =
\frac{1}{N}
\sum_{i=1}^{N}g_i
\]

where \(N\) is the number of distributed workers.

Each process then applies the synchronized gradient to its local model.

### Important

This notebook requires a multi-GPU environment.

Google Colab commonly provides only one GPU, so the full multi-GPU DDP section is designed for a multi-GPU runtime or multiple-GPU machine.

A single GPU can still execute the synchronization code in world-size 1 for verification.

In [ ]:
# ============================================================
# DDP training script
# ============================================================

%%writefile ddp_train.py

import os
import time
import torch
import torch.distributed as dist
import torch.nn as nn

from torch.nn.parallel import DistributedDataParallel
from torch.utils.data import (
    DataLoader,
    TensorDataset,
    DistributedSampler
)


class Model(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.net(x)


def setup():

    dist.init_process_group(
        backend="nccl"
    )

    rank = dist.get_rank()

    local_rank = int(
        os.environ["LOCAL_RANK"]
    )

    torch.cuda.set_device(
        local_rank
    )

    return rank, local_rank


def main():

    rank, local_rank = setup()

    device = torch.device(
        f"cuda:{local_rank}"
    )

    # --------------------------------------------------------
    # Synthetic dataset
    # --------------------------------------------------------

    X = torch.randn(
        20000,
        128
    )

    Y = torch.randint(
        0,
        10,
        (20000,)
    )

    dataset = TensorDataset(
        X,
        Y
    )

    sampler = DistributedSampler(
        dataset,
        shuffle=True
    )

    loader = DataLoader(
        dataset,
        batch_size=128,
        sampler=sampler
    )

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = Model().to(device)

    model = DistributedDataParallel(
        model,
        device_ids=[local_rank]
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-3
    )

    loss_fn = nn.CrossEntropyLoss()

    start = time.time()

    for epoch in range(5):

        sampler.set_epoch(epoch)

        total_loss = 0

        for x, y in loader:

            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            output = model(x)

            loss = loss_fn(
                output,
                y
            )

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        # Synchronize all workers
        dist.barrier()

        if rank == 0:

            print(
                f"Epoch {epoch+1}/5 - "
                f"Loss: "
                f"{total_loss / len(loader):.4f}"
            )

    dist.barrier()

    elapsed = time.time() - start

    if rank == 0:

        print(
            f"Training time: "
            f"{elapsed:.2f} seconds"
        )

    dist.destroy_process_group()


if __name__ == "__main__":
    main()

In [ ]:
# ============================================================
# Check GPU configuration
# ============================================================

import torch

gpu_count = torch.cuda.device_count()

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", gpu_count)

for i in range(gpu_count):

    print(
        f"GPU {i}:",
        torch.cuda.get_device_name(i)
    )

if gpu_count >= 2:

    print(
        "\nMulti-GPU DDP can be launched."
    )

else:

    print(
        "\nOnly one GPU is available. "
        "Run this script on a multi-GPU "
        "runtime/machine for true DDP."
    )

In [ ]:
# ============================================================
# Launch DDP when multiple GPUs are available
# ============================================================

import subprocess
import sys

if torch.cuda.device_count() >= 2:

    command = [
        "torchrun",
        "--standalone",
        f"--nproc_per_node={torch.cuda.device_count()}",
        "ddp_train.py"
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    print(result.stdout)

    if result.stderr:
        print(result.stderr)

else:

    print(
        "DDP launch skipped because "
        "this runtime has fewer than 2 GPUs."
    )

In [ ]:
# ============================================================
# Parameter synchronization concept check
# ============================================================

import torch
import torch.distributed as dist

print(
    "DDP uses gradient all-reduction during "
    "backpropagation."
)

print(
    """
Worker 0 gradient ─┐
Worker 1 gradient ─┼──> All-Reduce ──> Averaged Gradient
Worker 2 gradient ─┤
Worker N gradient ─┘

              ↓

        Local optimizer
              ↓

      Synchronized models
    """
)

# Conclusion

A DistributedDataParallel training pipeline was constructed using PyTorch Distributed.

The implementation includes:

- Distributed process-group initialization
- NCCL backend
- GPU-to-process assignment
- DistributedSampler
- DistributedDataParallel
- Synchronized gradient updates
- Epoch-level synchronization barriers
- Distributed training benchmarking

DDP allows multiple GPUs to process different portions of the training dataset while synchronizing gradients between workers.

For \(N\) workers, the effective training process performs distributed gradient synchronization:

\[
g=
\frac{1}{N}
\sum_{i=1}^{N}g_i
\]

The approach scales deep-learning training across multiple GPUs while keeping model parameters synchronized.

A single-GPU Google Colab runtime cannot demonstrate true multi-GPU speedup; therefore, the notebook automatically detects the available GPU count and only launches the multi-process DDP experiment when at least two GPUs are available.